# Creative Ideas with Everyday Objects

Name an everyday object and ask GPT-6 Luna for one unusual, practical way to use it.


## Before you run

Run the next cell to install the OpenAI Python SDK. Then run the idea cell. It will ask for an OpenAI API key using a hidden prompt. Use an API key, not your ChatGPT password. The key is only held in this Colab runtime; it is not written into this notebook.


In [ ]:
%pip -q install openai


In [ ]:
from getpass import getpass
from openai import (
    APIConnectionError,
    APIStatusError,
    AuthenticationError,
    OpenAI,
    RateLimitError,
)

# Ask what object to use.
item = input("Name an everyday object: ").strip()
if not item:
    raise ValueError("Please enter an object and run this cell again.")

# getpass hides the API key as you type it.
api_key = getpass("Enter your OpenAI API key (input is hidden): ")
if not api_key.strip():
    raise ValueError("Please enter your API key and run this cell again.")

client = OpenAI(api_key=api_key)
del api_key  # Do not print or save the key.

try:
    response = client.responses.create(
        model="gpt-6-luna",
        instructions=(
            "Suggest exactly ONE unusual but practical use for the everyday object. "
            "Keep it safe, legal, realistic, and low-cost. Give it a short title "
            "and explain it in a few simple sentences. Do not list alternatives."
        ),
        input=f"Everyday object: {item}",
        reasoning={"effort": "low"},
        max_output_tokens=500,
    )

    # Check the text, response status, and whether the output limit stopped it.
    idea = (response.output_text or "").strip()
    status = response.status or "unknown"
    details = response.incomplete_details
    reason = getattr(details, "reason", None) if details else None

    if idea:
        print(idea)
        if status == "incomplete" and reason == "max_output_tokens":
            print("\nThe answer may be cut off. Run the cell again for a complete idea.")
    elif status == "incomplete" and reason == "max_output_tokens":
        print("No idea text came back because the response reached its output limit. Please run the cell again.")
    elif status != "completed":
        print(f"No idea was returned (response status: {status}). Please check your connection and try again.")
    else:
        print("The request completed, but the API returned no text. Please run the cell again.")
except AuthenticationError:
    print("Authentication failed. Check that you entered a valid OpenAI API key (not your ChatGPT password), then rerun this cell.")
except RateLimitError:
    print("The API usage limit was reached. Check your OpenAI API usage and billing, or try again later.")
except APIConnectionError:
    print("Could not connect to OpenAI. Check your internet connection, then rerun this cell.")
except APIStatusError as error:
    print(f"OpenAI returned an HTTP {error.status_code} error. Check your API access and try again.")
except Exception as error:
    # Do not print the raw error: it could contain request details.
    print(f"Something unexpected happened ({type(error).__name__}). Please check your entries and try again.")
